In [1]:
import os
import torch
import pandas as pd
import wandb

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import PeftModel
from trl import SFTTrainer, SFTConfig

from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from rouge import Rouge

/root/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## W&B Setting and Data Load

In [2]:
# W&B 세팅
wandb.init(project="SOLAR-Summarization", name="solar-10.7b-qlora-v1")

# Data Load
train_df = pd.read_csv('../Data/train_processed.csv')
dev_df = pd.read_csv('../Data/dev_processed.csv')

# 메모리에 train_dataset 이랑 eval_dataset 을 각인시키기
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(dev_df)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /data/ephemeral/home/.netrc.
wandb: Currently logged in as: gam10678 (gam10678-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 4-Bit 양자화 및 모델/토크나이저 Load

In [3]:
model_id = "Upstage/SOLAR-10.7B-Instruct-v1.0"

# 24GB VRAM 에 맞추기 위한 4bit 양자화 설정해보기
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("💻 4Bit Model 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Decoder-only 모델은 padding token 진행 필수
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"       # GPU 환경에 맞게 자동으로 Split 해서 Upload
)

# 학습을 위한 k-bit 모델을 준비 (Gradient Checkpointing 등 메모리 최적화)
model = prepare_model_for_kbit_training(model)

💻 4Bit Model 로드 중...


Loading weights: 100%|██████████| 435/435 [00:02<00:00, 158.67it/s]


## LoRA 설정

모델 전체가 아니라 '일부 가중치' 만 학습해보는 전략

In [4]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()      # 학습할 파라미터 비율 출력 (1~2% 내외 정도?)

trainable params: 62,914,560 || all params: 10,794,438,656 || trainable%: 0.5828


## 프롬프트 포맷팅

In [5]:
# KoBART 처럼 Encoder / Decoder 가 나뉘어 있지 않아서, 텍스트를 하나의 '지시문' 형태로 묶어줘야 함.
def formatting_prompts_func(example):
    output_texts = []

    # 데이터가 리스트면 그대로 쓰고, 문자열이면 강제로 리스트로 묶어주기
    dialogues = example['dialogue'] if isinstance(example['dialogue'], list) else [example['dialogue']]
    summaries = example['summary'] if isinstance(example['summary'], list) else [example['summary']]

    # zip으로 묶어서 안전하게 프롬프트를 생성
    for d, s in zip(dialogues, summaries):
        text = f"다음 대화를 1~2문장으로 요약하시오.\n\n[대화]\n{d}\n\n[요약]\n{s}{tokenizer.eos_token}"
        output_texts.append(text)
        
    return output_texts

## SFTTrainer 학습 세팅

In [ ]:
tokenizer.model_max_length = 1024       # 운동 갔다와서 얘 max 값 생각해보기

# 훈련 세팅전에 미리 완벽한 프롬프트로 깍기 (자꾸 List 및 Type 에러를 반환해서 에러 해결책으로)
def create_prompt(example):
    # 한 줄 한 줄 합쳐서 'text' 라는 컬럼으로 반환
    text = f"다음 대화를 1~2문장으로 요약하시오.\n\n[대화]\n{example['dialogue']}\n\n[요약]\n{example['summary']}{tokenizer.eos_token}"

    return {"text": text}

# dataset.map() 을 통해 train 과 eval 데이터셋에 'text' 컬럼을 영구적으로 박기
train_dataset = train_dataset.map(create_prompt)
eval_dataset = eval_dataset.map(create_prompt)

Map: 100%|██████████| 499/499 [00:00<00:00, 20332.61 examples/s]


In [7]:
sft_config = SFTConfig(
    output_dir="./Model_Save/solar_qlora_checkpoints",
    num_train_epochs=2,                 # 모델이 크니까 Epoch 1 ~ 2 도 충분
    per_device_train_batch_size=2,      # VRAM 을 위해서 최소한으로 Set (메모리 이슈 발생시 1로 임의조정 필요)     
    gradient_accumulation_steps=4,      # 실질적인 배치 사이즈를 8(2*4) 로 늘리기
    optim="paged_adamw_8bit",           # 메모리 효율적인 옵티마이저
    learning_rate=2e-5,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    logging_steps=50,
    report_to='wandb',
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    processing_class=tokenizer         # tokenizer라는 이름 대신 processing_class 
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Truncating eval dataset: 100%|██████████| 499/499 [00:00<00:00, 168841.38 examples/s]
[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Train

In [8]:
print("SOLAR-10.7B QLoRA 학습 시작")
trainer.train()

SOLAR-10.7B QLoRA 학습 시작


Epoch,Training Loss,Validation Loss
1,0.749257,0.755453
2,0.700516,0.749165


TrainOutput(global_step=3116, training_loss=0.7606920501233984, metrics={'train_runtime': 31429.8557, 'train_samples_per_second': 0.793, 'train_steps_per_second': 0.099, 'total_flos': 9.882799472820142e+17, 'train_loss': 0.7606920501233984})

## Save

In [11]:
best_model_path = "./Model_Save/solar_best_lora"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
wandb.finish()

print(f"학습 완료 {best_model_path} 에 저장되었습니다.")

학습 완료 ./Model_Save/solar_best_lora 에 저장되었습니다.


## Score 측정

In [2]:
# Model Load
base_model_id = "upstage/SOLAR-10.7B-Instruct-v1.0"
lora_model_path = "./Model_Save/solar_best_lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, lora_model_path)
model.eval()

Loading weights: 100%|██████████| 435/435 [00:02<00:00, 160.03it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=2)
        (layers): ModuleList(
          (0-47): 48 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [3]:
# 1. 'Test' 가 아니라 'Dev(검증)' 데이터로 모의고사 측정
print("검증(Dev) 데이터로 로컬 ROUGE 점수를 측정합니다.")
dev_df = pd.read_csv('../Data/dev_processed.csv')

predicted_summaries = []
actual_summaries = dev_df['summary'].tolist()       # 정답지

for dialogue in tqdm(dev_df['dialogue']):
    prompt = f"다음 대화를 1~2문장으로 요약하시오.\n\n[대화]\n{dialogue}\n\n[요약]\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            repetition_penalty=1.2,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    summary_only = generated_text.replace(prompt, "").strip()

    # 예외 처리 : 모델이 가끔 빈 칸을 뱉으면 ROUGE 계산 시 에러가 나므로 점(.) 으로 대체
    if len(summary_only) == 0:
        summary_only = "."
    
    predicted_summaries.append(summary_only)

검증(Dev) 데이터로 로컬 ROUGE 점수를 측정합니다.


100%|██████████| 499/499 [3:00:09<00:00, 21.66s/it]  


In [4]:
# 2. ROUGE 점수 채점 및 결과 출력
rouge = Rouge()

# avg=True 를 주면 전체 데이터의 평균 점수를 예쁘게 뽑아준다.
scores = rouge.get_scores(predicted_summaries, actual_summaries, avg=True)
final_result = (scores['rouge-1']['f'] + scores['rouge-2']['f'] + scores['rouge-l']['f']) / 3.0 * 100.0

print("\n🏆 [로컬 ROUGE 점수 결과] 🏆")
print(f"ROUGE-1: {scores['rouge-1']['f']:.4f}")
print(f"ROUGE-2: {scores['rouge-2']['f']:.4f}")
print(f"ROUGE-L: {scores['rouge-l']['f']:.4f}")
print(f"Final Result: {final_result:.2f}")


🏆 [로컬 ROUGE 점수 결과] 🏆
ROUGE-1: 0.1859
ROUGE-2: 0.0516
ROUGE-L: 0.1781
Final Result: 13.85


## Infer & Submission

In [5]:
# Test Data 및 원본/LoRA Model Load
test_df = pd.read_csv('../Data/test_processed.csv')
sample_submission = pd.read_csv('../Data/sample_submission.csv')

In [6]:
# 1. 추론 시작
predicted_summaries = []

for dialogue in tqdm(test_df['dialogue']):
    # 훈련할 때 썼던 프롬프트와 100% 똑같은 형식을 써야 됨
    prompt = f"다음 대화를 1~2문장으로 요약하시오.\n\n[대화]\n{dialogue}\n\n[요약]\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Inference Tuning 파라미터 적용 (생성 길이, 페널티 등)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,                 # 요약문 100 토큰 부여
            repetition_penalty=1.2,             # 같은 말 반복 방지
            temperature=0.1,                    # ㅇ요약은 창의성 보다 팩트 위주니까 온도 하강
            pad_token_id=tokenizer.eos_token_id
        )

    # 출력된 텍스트에서 프롬프트 부분은 잘라내고 순수 요약 부분만 추출
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    summary_only = generated_text.replace(prompt, "").strip()
    predicted_summaries.append(summary_only)

100%|██████████| 499/499 [3:14:10<00:00, 23.35s/it]  


In [7]:
# Submission.csv 생성 및 저장
sample_submission['summary'] = predicted_summaries
sample_submission.to_csv('../Data/solar_submission_v1.csv', index=False, encoding='utf-8-sig')

print("추론 완료 '../Data/solar_submission_v1.csv' 파일이 생성되었습니다.")

추론 완료 '../Data/solar_submission_v1.csv' 파일이 생성되었습니다.
